# GAVD: extract features + train Random Forest classifier

## 1.  Setup

In [1]:
# Configure matplotlib FIRST before any other imports
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt

# Verify matplotlib backend
print(f"Matplotlib backend: {matplotlib.get_backend()}")

# Enable interactive mode
plt.ion()

Matplotlib backend: module://matplotlib_inline.backend_inline


In [2]:
# Suppress TensorFlow and MediaPipe logs
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['GLOG_minloglevel'] = '3'

import sys
import string
import contextlib
import pandas as pd
import seaborn as sns
import numpy as np
from pathlib import Path
from collections import defaultdict

# Setup paths
project_root = Path.cwd().parent.parent
video_base_path = project_root / "data" / "youtube"
data_root = project_root / "experiments" / "exp3" / "data"

sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Data root: {data_root}")
print(f"Video path: {video_base_path}")

Project root: /Users/pmui/dev/alex/alexpose
Data root: /Users/pmui/dev/alex/alexpose/experiments/exp3/data
Video path: /Users/pmui/dev/alex/alexpose/data/youtube


In [3]:
from ambient.gavd import GAVDDataLoader
from ambient.pose.keypoint_extractor import SequenceKeypointExtractor
from ambient.pose.joint_angles import get_joint_angles

from ambient.classification.rf_classifier import (
    RFGaitClassifier,
    RFClassifierConfig,
    GaitFeatureVector
)

print("✓ Imports successful")

✓ Imports successful


## 2. Explore Training Dataset

In [4]:
data_root

PosixPath('/Users/pmui/dev/alex/alexpose/experiments/exp3/data')

In [5]:
# Get condition directories
condition_paths = [
    p for p in data_root.iterdir() 
    if p.is_dir() and p.name[0] in string.ascii_letters
]

len(condition_paths)

3

In [6]:

print(f"Found {len(condition_paths)} conditions:\n")
for path in sorted(condition_paths):
    csv_files = list(path.glob("*.csv"))
    print(f"  {path.name:15s}: {len(csv_files)} CSV files")

Found 3 conditions:

  normal         : 12 CSV files
  parkinsons     : 9 CSV files
  stroke         : 12 CSV files


## 3.  Extract Keypoints from One Gait (CSV) Sequence

In [7]:
normal_path = condition_paths[2]
stroke_path = condition_paths[0]
parkinsons_path = condition_paths[1]

normal_path, stroke_path, parkinsons_path

(PosixPath('/Users/pmui/dev/alex/alexpose/experiments/exp3/data/normal'),
 PosixPath('/Users/pmui/dev/alex/alexpose/experiments/exp3/data/stroke'),
 PosixPath('/Users/pmui/dev/alex/alexpose/experiments/exp3/data/parkinsons'))

In [8]:
# Let's get the first CSV from the normal_path
normal_csv = list(normal_path.glob("*.csv"))[7]
stroke_csv = list(stroke_path.glob("*.csv"))[0]
parkinsons_csv = list(parkinsons_path.glob("*.csv"))[3]
normal_csv, stroke_csv, parkinsons_csv

(PosixPath('/Users/pmui/dev/alex/alexpose/experiments/exp3/data/normal/cljo30lnz001q3n6lopfty7q5.csv'),
 PosixPath('/Users/pmui/dev/alex/alexpose/experiments/exp3/data/stroke/cljvvsucg00043n6l4evgn7q4.csv'),
 PosixPath('/Users/pmui/dev/alex/alexpose/experiments/exp3/data/parkinsons/cljnz5sb1000o3n6lntosswwz.csv'))

In [9]:
gavd_loader = GAVDDataLoader()

normal_df = gavd_loader.load_gavd_data(normal_csv)
stroke_df = gavd_loader.load_gavd_data(stroke_csv)
parkinsons_df = gavd_loader.load_gavd_data(parkinsons_csv)

normal_df.shape, stroke_df.shape, parkinsons_df.shape

((80, 12), (166, 12), (81, 12))

In [10]:
normal_sid = normal_df.iloc[0]["seq"]
stroke_sid = stroke_df.iloc[0]["seq"]
parkinsons_sid = parkinsons_df.iloc[0]["seq"]

normal_sid, stroke_sid, parkinsons_sid

('cljo30lnz001q3n6lopfty7q5',
 'cljvvsucg00043n6l4evgn7q4',
 'cljnz5sb1000o3n6lntosswwz')

In [11]:
video_base_path

PosixPath('/Users/pmui/dev/alex/alexpose/data/youtube')

In [12]:
extractor = SequenceKeypointExtractor()

In [ ]:
# Extract with automatic filtering - only frames with visible person
normal_keypoints_array = extractor.extract_from_sequence(
    sequence_data=normal_df,
    video_base_path=video_base_path,
    verbose=True,
    filter_empty=True,      # ← Remove frames with < 25 keypoints
    min_keypoints=25        # ← Require at least 25 out of 33 keypoints
)

stroke_keypoints_array = extractor.extract_from_sequence(
    sequence_data=stroke_df,
    video_base_path=video_base_path,
    verbose=True,
    filter_empty=True,
    min_keypoints=25
)

parkinsons_keypoints_array = extractor.extract_from_sequence(
    sequence_data=parkinsons_df,
    video_base_path=video_base_path,
    verbose=True,
    filter_empty=True,
    min_keypoints=25
)

# Print statistics for each
print("\n" + "="*70)
print("EXTRACTION SUMMARY")
print("="*70)
extractor.print_extraction_statistics(normal_keypoints_array, "Normal Gait")
extractor.print_extraction_statistics(stroke_keypoints_array, "Stroke Gait")
extractor.print_extraction_statistics(parkinsons_keypoints_array, "Parkinsons Gait")

I0000 00:00:1769065643.842573 17312858 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2 Max
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1769065643.893157 17312860 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769065643.912862 17312865 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
2026-01-21 23:07:23.913 | INFO     | ambient.pose.keypoint_extractor:_process_all_frames:416 - Processing sequence: cljo30lnz001q3n6lopfty7q5 (80 frames)


	frame 311 (1/80)


W0000 00:00:1769065644.810598 17312860 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


	frame 321 (11/80)
	frame 331 (21/80)
	frame 341 (31/80)
	frame 351 (41/80)
	frame 361 (51/80)


In [ ]:
len(normal_keypoints_array[0].keypoints), len(stroke_keypoints_array[0].keypoints), len(parkinsons_keypoints_array[0].keypoints)

In [ ]:
# let's look at the first keypoint at frame 0
normal_keypoints_array[0].keypoints[0]

In [ ]:
from pprint import pprint

print(vars(normal_keypoints_array[0].keypoints[0]))

## 4.  Calculate Frame-by-Frame Joint Angles

In [ ]:
normal_joint_angles = get_joint_angles(
    keypoints_array=normal_keypoints_array,
    keypoint_format="BLAZEPOSE_33",
    fps=30.0,
    confidence_threshold=0.3,
    sequence_id=normal_sid
)
stroke_joint_angles = get_joint_angles(
    keypoints_array=stroke_keypoints_array,
    keypoint_format="BLAZEPOSE_33",
    fps=30.0,
    confidence_threshold=0.3,
    sequence_id=stroke_sid
)
parkinsons_joint_angles = get_joint_angles(
    keypoints_array=parkinsons_keypoints_array,
    keypoint_format="BLAZEPOSE_33",
    fps=30.0,
    confidence_threshold=0.3,
    sequence_id=parkinsons_sid
)

# let's see which joint angles are calculated for Frame 0 (same for all other frames)
print(normal_joint_angles.frames[0].angles.keys())

In [ ]:
print("Normal Gait: Left Hip")
normal_joint_angles.get_statistics("left_hip")

In [ ]:
print("Stroke Gait: Left Hip")
stroke_joint_angles.get_statistics("left_hip")


In [ ]:
print("Parkinsons Gait: Left Hip")
parkinsons_joint_angles.get_statistics("left_hip")

Let's visualize the joint angle changes over a gait sequence

In [ ]:
import matplotlib.pyplot as plt

def plot_joint_angles_statistics(name, joint_angles):

    print(f"\nVisualizing joint angles for {name}...")
    print(f"Number of frames with joint angles: {len(joint_angles.frames)}")

    if len(joint_angles.frames) > 0 and len(joint_angles.frames[0].angles) > 0:
        joints = ['left_hip', 'left_knee', 'left_ankle', 'right_hip', 'right_knee', 'right_ankle']
        
        # Create the figure
        fig, axes = plt.subplots(2, 3, figsize=(16, 10))
        fig.suptitle(f"{name} Joint Angles Over Time - {normal_path.name}", fontsize=18, fontweight='bold')
        
        for idx, joint_name in enumerate(joints):
            row = idx // 3
            col = idx % 3
            ax = axes[row, col]
            
            try:
                # Get angle data
                angles = joint_angles.get_joint_angle_series(joint_name)
                stats = joint_angles.get_statistics(joint_name)
                
                # Check if we have valid data
                if stats['valid_count'] > 0 and not np.isnan(stats['mean']):
                    # Filter out NaN values
                    valid_mask = ~np.isnan(angles)
                    valid_frames = np.where(valid_mask)[0]
                    valid_angles = angles[valid_mask]
                    
                    if len(valid_angles) > 0:
                        # Plot the angle series
                        ax.plot(valid_frames, valid_angles, 'b-', linewidth=2, alpha=0.7, label='Angle')
                        
                        # Add mean line
                        ax.axhline(y=stats['mean'], color='red', linestyle='--', 
                                linewidth=2, alpha=0.8, label=f"Mean: {stats['mean']:.1f}°")
                        
                        # Add shaded region for std
                        ax.axhspan(stats['mean'] - stats['std'], stats['mean'] + stats['std'], 
                                alpha=0.2, color='red', label=f"±1 SD")
                        
                        # Formatting
                        ax.set_title(joint_name.replace('_', ' ').title(), 
                                    fontsize=13, fontweight='bold', pad=10)
                        ax.set_xlabel('Frame Number', fontsize=11)
                        ax.set_ylabel('Angle (degrees)', fontsize=11)
                        ax.legend(loc='best', fontsize=9, framealpha=0.9)
                        ax.grid(True, alpha=0.3, linestyle=':', linewidth=0.5)
                        
                        # Set reasonable y-axis limits
                        y_margin = (stats['max'] - stats['min']) * 0.1
                        ax.set_ylim(stats['min'] - y_margin, stats['max'] + y_margin)
                    else:
                        ax.text(0.5, 0.5, 'No valid angle data', 
                            ha='center', va='center', transform=ax.transAxes, 
                            fontsize=12, color='gray')
                        ax.set_title(joint_name.replace('_', ' ').title())
                else:
                    ax.text(0.5, 0.5, 'Insufficient data', 
                        ha='center', va='center', transform=ax.transAxes, 
                        fontsize=12, color='gray')
                    ax.set_title(joint_name.replace('_', ' ').title())
                    
            except Exception as e:
                ax.text(0.5, 0.5, f'Error: {str(e)[:40]}', 
                    ha='center', va='center', transform=ax.transAxes, 
                    fontsize=10, color='red', wrap=True)
                ax.set_title(joint_name.replace('_', ' ').title())
                print(f"Error plotting {joint_name}: {e}")
        
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()
        
        # Print statistics table
        print("\n" + "="*70)
        print("Joint Angle Statistics")
        print("="*70)
        print(f"{'Joint':<15} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8} {'Range':>8}")
        print("-"*70)
        
        for joint_name in joints:
            try:
                stats = joint_angles.get_statistics(joint_name)
                if stats['valid_count'] > 0:
                    print(f"{joint_name:<15} {stats['mean']:>8.1f}° {stats['std']:>7.1f}° "
                        f"{stats['min']:>7.1f}° {stats['max']:>7.1f}° {stats['range']:>7.1f}°")
                else:
                    print(f"{joint_name:<15} {'No data':>8}")
            except Exception as e:
                print(f"{joint_name:<15} Error: {e}")
        print("="*70 + "\n")
        
    else:
        print("\n⚠️  WARNING: No joint angles computed!")
        print(f"   Total frames: {len(joint_angles.frames)}")
        if len(joint_angles.frames) > 0:
            print(f"   Angles in first frame: {len(joint_angles.frames[0].angles)}")
            print(f"   Available joints: {list(joint_angles.frames[0].angles.keys())}")
        print("   Check if keypoints were extracted successfully.\n")

In [ ]:
plot_joint_angles_statistics("Normal Gait", normal_joint_angles)

In [ ]:
plot_joint_angles_statistics("Stroke Gait", stroke_joint_angles)

In [ ]:
plot_joint_angles_statistics("Parkinsons Gait", parkinsons_joint_angles)

## 5.  Extract Features for Classification

In [ ]:
from ambient.classification.knn_classifier import (
    KNNGaitClassifier,
    KNNClassifierConfig,
    GaitFeatureVector
)

In [ ]:
# Create feature vector
feature_vector = GaitFeatureVector.from_joint_angles(
    normal_joint_angles,
    sample_id=normal_sid,
    condition_label="normal"
)

print("Feature Vector:")
print(f"  Sample ID: {feature_vector.sample_id}")
print(f"  Condition: {feature_vector.condition_label}")
print(f"\nFeatures:")
for name in GaitFeatureVector.get_feature_names():
    value = getattr(feature_vector, name)
    print(f"  {name:20s}: {value:7.2f}")

In [ ]:
def extract_all_features(condition_paths, video_base_path):
    """Extract features from all condition directories."""
    all_features = []
    condition_counts = defaultdict(int)
    gavd_loader = GAVDDataLoader()
    
    for condition_path in condition_paths:
        condition_name = condition_path.name
        print(f"\nProcessing: {condition_name}")
        
        for csv_path in condition_path.glob("*.csv"):
            try:
                df = gavd_loader.load_gavd_data(str(csv_path))
                sequences = gavd_loader.organize_by_sequence(df)
                
                for seq_id in sequences:
                    try:
                        sequence_df = sequences[seq_id]
                        
                        # Extract keypoints
                        extractor = SequenceKeypointExtractor()
                        keypoints_array = extractor.extract_from_sequence(
                            sequence_df,
                            video_base_path=video_base_path,
                            verbose=False
                        )
                        
                        if not keypoints_array:
                            continue
                        
                        # Calculate joint angles
                        joint_angles = get_joint_angles(
                            keypoints_array=keypoints_array,
                            keypoint_format="BLAZEPOSE_33",
                            fps=30.0,
                            confidence_threshold=0.3,
                            sequence_id=seq_id
                        )
                        
                        if len(joint_angles.frames) == 0:
                            continue
                        
                        # Create feature vector
                        feature_vector = GaitFeatureVector.from_joint_angles(
                            joint_angles,
                            sample_id=seq_id,
                            condition_label=condition_name
                        )
                        
                        all_features.append(feature_vector)
                        condition_counts[condition_name] += 1
                        print(f"  ✓ {seq_id}")
                    
                    except Exception as e:
                        print(f"  ✗ {seq_id}: {e}")
            
            except Exception as e:
                print(f"  Error processing {csv_path.name}: {e}")
    
    return all_features, dict(condition_counts)

In [ ]:
# Extract all features
all_features, condition_counts = extract_all_features(condition_paths, video_base_path)

print(f"\n{'='*60}")
print(f"Total features extracted: {len(all_features)}")
print(f"\nCondition distribution:")
for condition, count in sorted(condition_counts.items()):
    print(f"  {condition:15s}: {count:3d} samples")

### Save and Load Feature Data

Let's create functions to save and load the extracted features to avoid re-extracting them every time.

In [ ]:
from ambient.utils.features import save_features, load_features

save_path = save_features(all_features, condition_counts)

print(f"\nFeatures saved to: {save_path}")

### Load Previously Extracted Features

To load previously saved features instead of re-extracting them, uncomment and run the cell below:

In [ ]:
all_features, condition_counts = load_features()
print(f"\n{'='*60}")
print(f"Total features loaded: {len(all_features)}")
print(f"\nCondition distribution:")
for condition, count in sorted(condition_counts.items()):
     print(f"  {condition:15s}: {count:3d} samples")

## 5. Visualize Feature Distributions

Let's visualize how features differ across conditions.

In [ ]:
# Convert to DataFrame for visualization
feature_data = []
for fv in all_features:
    row = {name: getattr(fv, name) for name in GaitFeatureVector.get_feature_names()}
    row['condition'] = fv.condition_label
    feature_data.append(row)

df_features = pd.DataFrame(feature_data)
print(df_features.head())

In [ ]:
# Plot asymmetry features by condition
if len(all_features) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Asymmetry Features by Condition", fontsize=16, fontweight='bold')
    
    asymmetry_features = ['hip_asymmetry', 'knee_asymmetry', 'ankle_asymmetry']
    
    for idx, feature in enumerate(asymmetry_features):
        ax = axes[idx]
        df_features.boxplot(column=feature, by='condition', ax=ax)
        ax.set_title(feature.replace('_', ' ').title(), fontsize=12)
        ax.set_xlabel('Condition', fontsize=11)
        ax.set_ylabel('Asymmetry (degrees)', fontsize=11)
        plt.sca(ax)
        plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
else:
    print("No features to visualize")

In [ ]:
# Plot mean joint angles by condition
if len(all_features) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle("Mean Joint Angles by Condition", fontsize=16, fontweight='bold')
    
    angle_features = [
        'left_hip_mean', 'left_knee_mean', 'left_ankle_mean',
        'right_hip_mean', 'right_knee_mean', 'right_ankle_mean'
    ]
    
    for idx, feature in enumerate(angle_features):
        ax = axes[idx // 3, idx % 3]
        df_features.boxplot(column=feature, by='condition', ax=ax)
        ax.set_title(feature.replace('_', ' ').title(), fontsize=12)
        ax.set_xlabel('Condition', fontsize=11)
        ax.set_ylabel('Angle (degrees)', fontsize=11)
        plt.sca(ax)
        plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
else:
    print("No features to visualize")

## 6. Train & Eval Random Forest Classifier

Now let's train a RF classifier on our extracted features.

In [ ]:
# Split into train/test (80/20)
np.random.seed(42)
indices = np.random.permutation(len(all_features))
split_idx = int(0.8 * len(all_features))

train_indices = indices[:split_idx]
test_indices = indices[split_idx:]

train_features = [all_features[i] for i in train_indices]
test_features = [all_features[i] for i in test_indices]

print(f"Training samples: {len(train_features)}")
print(f"Test samples: {len(test_features)}")

In [ ]:

# Configure and train classifier
config = KNNClassifierConfig(
    n_neighbors=5,
    weights="distance",
    metric="euclidean",
    normalize_features=True
)

classifier = KNNGaitClassifier(config=config)
metrics = classifier.train(train_features, validate=True)

print("\nTraining Results:")
print(f"  Training Accuracy: {metrics['train_accuracy']:.3f}")
if 'cv_mean_accuracy' in metrics:
    print(f"  CV Accuracy: {metrics['cv_mean_accuracy']:.3f} ± {metrics['cv_std_accuracy']:.3f}")
print(f"  Classes: {metrics['classes']}")

In [ ]:
# Evaluate on test set
eval_metrics = classifier.evaluate(test_features)

print(f"Test Accuracy: {eval_metrics['accuracy']:.3f}")
print(f"\nClassification Report:")
report = eval_metrics['classification_report']
for class_name in eval_metrics['classes']:
    if class_name in report:
        cm = report[class_name]
        print(f"\n{class_name}:")
        print(f"  Precision: {cm['precision']:.3f}")
        print(f"  Recall: {cm['recall']:.3f}")
        print(f"  F1-Score: {cm['f1-score']:.3f}")

## 7.  Train & Eval Random Forest Classifier

In [ ]:
from ambient.classification.rf_classifier import RFClassifierConfig, RFGaitClassifier

# Configure and train Random Forest classifier
# Random Forest is an ensemble learning method that uses multiple decision trees
# to improve classification accuracy and reduce overfitting.
#
# Key parameters:
#   - n_estimators: Number of trees in the forest (more trees = better accuracy but slower)
#   - max_depth: Maximum depth of each tree (None = unlimited, prevents overfitting)
#   - min_samples_split: Minimum samples required to split a node (higher = more conservative)
#   - min_samples_leaf: Minimum samples required at leaf node (higher = smoother decision boundaries)
#   - max_features: Number of features to consider for best split ('sqrt' is recommended)
#   - class_weight: 'balanced' automatically adjusts for imbalanced datasets
#   - normalize_features: Standardize features to zero mean and unit variance
#   - random_state: Seed for reproducibility
config = RFClassifierConfig(
    n_estimators=100,           # Number of decision trees in the forest
    max_depth=None,             # No limit on tree depth (trees grow until pure)
    min_samples_split=2,        # Minimum samples to split an internal node
    min_samples_leaf=1,         # Minimum samples required at a leaf node
    max_features="sqrt",        # Use sqrt(n_features) for each split
    bootstrap=True,             # Use bootstrap sampling for building trees
    class_weight="balanced",    # Handle imbalanced classes automatically
    normalize_features=True,    # Standardize features before training
    random_state=42             # For reproducible results
)

# Initialize and train the classifier
classifier = RFGaitClassifier(config=config)
metrics = classifier.train(train_features, validate=True)

# Display training results
print("\n" + "="*70)
print("RANDOM FOREST TRAINING RESULTS")
print("="*70)
print(f"Training Accuracy:     {metrics['train_accuracy']:.3f}")
print(f"Number of Samples:     {metrics['n_samples']}")
print(f"Number of Features:    {metrics['n_features']}")
print(f"Number of Trees:       {metrics['n_estimators']}")
print(f"Classes:               {metrics['classes']}")

# Display cross-validation results if available
if 'cv_mean_accuracy' in metrics:
    print(f"\nCross-Validation Results:")
    print(f"  Mean Accuracy:       {metrics['cv_mean_accuracy']:.3f}")
    print(f"  Std Deviation:       {metrics['cv_std_accuracy']:.3f}")

# Display class distribution
if 'class_distribution' in metrics:
    print(f"\nClass Distribution:")
    for cls, count in metrics['class_distribution'].items():
        print(f"  {cls:15s}: {count:3d} samples")

# Display top 5 most important features
print(f"\nTop 5 Most Important Features:")
for feat_info in metrics['feature_importances'][:5]:
    print(f"  {feat_info['rank']}. {feat_info['feature']:20s}: {feat_info['importance']:.4f}")
print("="*70)

In [ ]:
# Evaluate on test set
eval_metrics = classifier.evaluate(test_features)

print(f"Test Accuracy: {eval_metrics['accuracy']:.3f}")
print(f"\nClassification Report:")
report = eval_metrics['classification_report']
for class_name in eval_metrics['classes']:
    if class_name in report:
        cm = report[class_name]
        print(f"\n{class_name}:")
        print(f"  Precision: {cm['precision']:.3f}")
        print(f"  Recall: {cm['recall']:.3f}")
        print(f"  F1-Score: {cm['f1-score']:.3f}")

## 8.  Train & Eval an SVM Classifier

In [ ]:
from ambient.classification.svm_classifier import (
    SVMGaitClassifier,
    SVMClassifierConfig
)

# Configure and train SVM classifier
# Support Vector Machine (SVM) is a powerful classifier that finds optimal
# decision boundaries (hyperplanes) in high-dimensional feature spaces.
#
# Key parameters:
#   - kernel: Type of kernel function ('rbf', 'linear', 'poly', 'sigmoid')
#   - C: Regularization parameter (higher = less regularization, may overfit)
#   - gamma: Kernel coefficient for 'rbf', 'poly', 'sigmoid' ('scale' is recommended)
#   - class_weight: 'balanced' automatically adjusts for imbalanced datasets
#   - probability: Enable probability estimates for predictions
#   - random_state: Seed for reproducibility
svm_config = SVMClassifierConfig(
    kernel='rbf',               # Radial Basis Function kernel for non-linear patterns
    C=10.0,                     # Regularization parameter
    gamma='scale',              # Kernel coefficient (1 / (n_features * X.var()))
    class_weight='balanced',    # Handle imbalanced classes automatically
    probability=True,           # Enable probability estimates
    random_state=42             # For reproducible results
)

# Initialize and train the SVM classifier
svm_classifier = SVMGaitClassifier(svm_config)
print("✓ SVM classifier created")

In [ ]:
# Train SVM classifier
svm_metrics = svm_classifier.train(train_features, validate=True)

# Display training results
print("\n" + "="*70)
print("SVM TRAINING RESULTS")
print("="*70)
print(f"Training Accuracy:     {svm_metrics['train_accuracy']:.3f}")
print(f"Number of Samples:     {svm_metrics['n_samples']}")
print(f"Number of Features:    {svm_metrics['n_features']}")
print(f"Classes:               {svm_metrics['classes']}")

# Display support vector information
if 'n_support_vectors' in svm_metrics:
    print(f"\nSupport Vector Information:")
    print(f"  Total Support Vectors: {svm_metrics['n_support_vectors']}")
    print(f"  Support Vectors per Class:")
    for cls, count in svm_metrics['support_vectors_per_class'].items():
        print(f"    {cls:15s}: {count:3d} vectors")

# Display cross-validation results if available
if 'cv_mean_accuracy' in svm_metrics:
    print(f"\nCross-Validation Results:")
    print(f"  Mean Accuracy:       {svm_metrics['cv_mean_accuracy']:.3f}")
    print(f"  Std Deviation:       {svm_metrics['cv_std_accuracy']:.3f}")

# Display class distribution
if 'class_distribution' in svm_metrics:
    print(f"\nClass Distribution:")
    for cls, count in svm_metrics['class_distribution'].items():
        print(f"  {cls:15s}: {count:3d} samples")

print("="*70)

In [ ]:
# Evaluate SVM on test set
svm_eval_metrics = svm_classifier.evaluate(test_features)

print(f"Test Accuracy: {svm_eval_metrics['accuracy']:.3f}")
print(f"\nClassification Report:")
svm_report = svm_eval_metrics['classification_report']
for class_name in svm_eval_metrics['classes']:
    if class_name in svm_report:
        cm = svm_report[class_name]
        print(f"\n{class_name}:")
        print(f"  Precision: {cm['precision']:.3f}")
        print(f"  Recall: {cm['recall']:.3f}")
        print(f"  F1-Score: {cm['f1-score']:.3f}")

In [ ]:
# Plot SVM confusion matrix
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 8))
sns.heatmap(
    svm_eval_metrics['confusion_matrix'],
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=svm_eval_metrics['classes'],
    yticklabels=svm_eval_metrics['classes']
)
plt.title('SVM Classifier - Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

print("\n✓ SVM classifier trained and evaluated")

## 9.  Train & Eval an XGBoost Classifier

In [ ]:
from ambient.classification.xgboost_classifier import (
    XGBoostGaitClassifier,
    XGBoostClassifierConfig
)

# Configure and train XGBoost classifier
# XGBoost (eXtreme Gradient Boosting) is a state-of-the-art gradient boosting
# algorithm that builds trees sequentially, with each tree correcting errors
# from previous trees. Often achieves the best accuracy on structured data.
#
# Key parameters:
#   - n_estimators: Number of boosting rounds (trees to build sequentially)
#   - max_depth: Maximum depth of each tree (lower = less overfitting)
#   - learning_rate: Step size shrinkage (lower = more conservative, needs more trees)
#   - subsample: Fraction of samples used for each tree (< 1.0 prevents overfitting)
#   - colsample_bytree: Fraction of features used for each tree
#   - reg_alpha: L1 regularization term (higher = more conservative)
#   - reg_lambda: L2 regularization term (higher = more conservative)
#   - random_state: Seed for reproducibility
xgb_config = XGBoostClassifierConfig(
    n_estimators=200,           # Number of boosting rounds
    max_depth=5,                # Maximum tree depth (shallower than RF)
    learning_rate=0.1,          # Learning rate (eta)
    subsample=0.8,              # Subsample ratio of training instances
    colsample_bytree=0.8,       # Subsample ratio of features
    reg_alpha=0.1,              # L1 regularization
    reg_lambda=1.0,             # L2 regularization
    min_child_weight=1,         # Minimum sum of instance weight in a child
    gamma=0.0,                  # Minimum loss reduction for split
    normalize_features=True,    # Standardize features before training
    random_state=42             # For reproducible results
)

# Initialize and train the XGBoost classifier
xgb_classifier = XGBoostGaitClassifier(xgb_config)
print("✓ XGBoost classifier created")

In [ ]:
# Train XGBoost classifier
xgb_metrics = xgb_classifier.train(train_features, validate=True)

# Display training results
print("\n" + "="*70)
print("XGBOOST TRAINING RESULTS")
print("="*70)
print(f"Training Accuracy:     {xgb_metrics['train_accuracy']:.3f}")
print(f"Number of Samples:     {xgb_metrics['n_samples']}")
print(f"Number of Features:    {xgb_metrics['n_features']}")
print(f"Number of Estimators:  {xgb_metrics['n_estimators']}")
print(f"Classes:               {xgb_metrics['classes']}")

# Display cross-validation results if available
if 'cv_mean_accuracy' in xgb_metrics:
    print(f"\nCross-Validation Results:")
    print(f"  Mean Accuracy:       {xgb_metrics['cv_mean_accuracy']:.3f}")
    print(f"  Std Deviation:       {xgb_metrics['cv_std_accuracy']:.3f}")

# Display class distribution
if 'class_distribution' in xgb_metrics:
    print(f"\nClass Distribution:")
    for cls, count in xgb_metrics['class_distribution'].items():
        print(f"  {cls:15s}: {count:3d} samples")

# Display top 5 most important features
if 'feature_importances' in xgb_metrics:
    print(f"\nTop 5 Most Important Features:")
    for feat_info in xgb_metrics['feature_importances'][:5]:
        print(f"  {feat_info['rank']}. {feat_info['feature']:20s}: {feat_info['importance']:.4f}")

print("="*70)

In [ ]:
# Evaluate XGBoost on test set
xgb_eval_metrics = xgb_classifier.evaluate(test_features)

print(f"Test Accuracy: {xgb_eval_metrics['accuracy']:.3f}")
print(f"\nClassification Report:")
xgb_report = xgb_eval_metrics['classification_report']
for class_name in xgb_eval_metrics['classes']:
    if class_name in xgb_report:
        cm = xgb_report[class_name]
        print(f"\n{class_name}:")
        print(f"  Precision: {cm['precision']:.3f}")
        print(f"  Recall: {cm['recall']:.3f}")
        print(f"  F1-Score: {cm['f1-score']:.3f}")

In [ ]:
# Plot XGBoost confusion matrix
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 8))
sns.heatmap(
    xgb_eval_metrics['confusion_matrix'],
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=xgb_eval_metrics['classes'],
    yticklabels=xgb_eval_metrics['classes']
)
plt.title('XGBoost Classifier - Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

print("\n✓ XGBoost classifier trained and evaluated")

## 10. Compare Classifier Performance

Let's compare the performance of all three classifiers side by side.

In [ ]:
# Compare classifier performance
import pandas as pd

# Collect metrics from all classifiers
comparison_data = {
    'Classifier': ['Random Forest', 'SVM', 'XGBoost'],
    'Test Accuracy': [
        eval_metrics['accuracy'],
        svm_eval_metrics['accuracy'],
        xgb_eval_metrics['accuracy']
    ]
}

# Add per-class metrics
for class_name in eval_metrics['classes']:
    if class_name in report:
        comparison_data[f'{class_name}_precision'] = [
            report[class_name]['precision'],
            svm_report[class_name]['precision'] if class_name in svm_report else 0,
            xgb_report[class_name]['precision'] if class_name in xgb_report else 0
        ]
        comparison_data[f'{class_name}_recall'] = [
            report[class_name]['recall'],
            svm_report[class_name]['recall'] if class_name in svm_report else 0,
            xgb_report[class_name]['recall'] if class_name in xgb_report else 0
        ]
        comparison_data[f'{class_name}_f1'] = [
            report[class_name]['f1-score'],
            svm_report[class_name]['f1-score'] if class_name in svm_report else 0,
            xgb_report[class_name]['f1-score'] if class_name in xgb_report else 0
        ]

df_comparison = pd.DataFrame(comparison_data)
print("\n" + "="*70)
print("CLASSIFIER PERFORMANCE COMPARISON")
print("="*70)
print(df_comparison.to_string(index=False))
print("="*70)

# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(comparison_data['Classifier']))
width = 0.35

ax.bar(x, comparison_data['Test Accuracy'], width, label='Test Accuracy', color='steelblue')
ax.set_xlabel('Classifier', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Classifier Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparison_data['Classifier'])
ax.legend()
ax.set_ylim([0, 1.0])
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, v in enumerate(comparison_data['Test Accuracy']):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✓ Classifier comparison complete")